In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
docs = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [5]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [6]:
query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

In [7]:
results = vectorstore.similarity_search(query, k=3)

In [9]:
results[0].page_content

'7.2 자격증 취득 지원\n직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다.\n자격 등급 축하금 (1 회성) 자격 수당 (월) 대상 자격증 예시\n기술사/기능장 200 만원 30 만원 금속재료, 용접, 기계가공 등\n기사 50 만원 10 만원 일반기계, 전기, 산업안전 등\n산업기사 30 만원 5 만원 기계설계, 위험물 등\n기능사 10 만원 3 만원 선반, 밀링, 특수용접 등\n\uf0b7 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수\n제한 없음.\n7.3 해외 연수 (Global Explorer)\n\uf0b7 대상: 연간 최우수 사원 (MVP) 및 우수 팀.\n\uf0b7 내용: 매년 10 월 독일/일본 등 선진 제조 현장 견학 및 문화 탐방 (7 박 9 일).\n\uf0b7 비용: 회사 전액 부담 (개인 경비 1,000 유로 별도 지급).\n8. 윤리 경영 및 보안 (Ethics & Security)\n8.1 직장 내 괴롭힘 방지 가이드'

In [12]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [13]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [14]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-3.1-flash-lite")

In [15]:
chain = prompt | llm | parser

In [16]:
response = chain.invoke({"context": results, "question": query})

In [ ]:
response # 지식(문서) 기반으로 답변!

'제공된 컨텍스트에 따르면, 국가 기술 자격 중 기사 자격증을 취득할 경우 다음과 같이 받을 수 있습니다.\n\n*   **축하금(1회성):** 50만 원\n*   **자격 수당(월):** 10만 원'

In [ ]:
response = llm.invoke(query) # RAG 파이프라인을 활용하지 않은 경우

In [ ]:
response # LLM이 학습한 일반적인 내용이 출력됨

AIMessage(content=[{'type': 'text', 'text': '국가기술자격증인 \'기사\' 자격증을 취득했을 때 받는 금전적 보상은 **\'자격 수당\'**과 **\'연봉 협상 시의 가치\'** 두 가지 측면에서 살펴봐야 합니다. 결론부터 말씀드리면 **"회사마다 다르지만, 통상적으로 월 5만 원에서 20만 원 내외의 수당을 받거나, 연봉 협상 시 긍정적인 요인으로 작용한다"**고 할 수 있습니다.\n\n상세한 내용은 다음과 같습니다.\n\n---\n\n### 1. 자격 수당 (매월 지급)\n많은 기업(특히 제조, 건설, 엔지니어링 업계)에서는 기술 자격증 소지자에게 매달 \'자격 수당\'을 지급합니다.\n*   **금액:** 회사 내규에 따라 다르지만, 보통 **기사 자격증 1개당 월 5만 원 ~ 20만 원** 정도가 가장 흔합니다.\n*   **지급 방식:** 기본급에 포함되거나, 별도 수당으로 매달 급여일에 지급됩니다.\n*   **주의점:** 모든 회사가 수당을 주지는 않습니다. 특히 자격증이 업무의 필수 요건(선임 필수)인 경우에는 수당을 따로 주지 않는 대신, 그 자격증이 있어야만 취업이 가능한 \'입사 조건\'으로 간주되기도 합니다.\n\n### 2. 연봉 협상 및 이직 시의 가치\n실제 연봉 자체를 올리는 가장 큰 효과는 자격 수당보다는 **\'연봉 협상\'**이나 **\'이직\'** 과정에서 나타납니다.\n*   **직무 전문성 입증:** 기사 자격증은 해당 분야의 이론적 지식과 실무 능력을 국가가 검증했다는 증거입니다. 경력직으로 이직할 때 동일 직무의 기사 자격증은 연봉을 협상할 때 "나는 이 분야의 전문가다"라는 강력한 근거가 됩니다.\n*   **채용 시 우대:** 기사 자격증 소지자는 신입 채용 시 가산점을 받거나, 서류 통과 확률이 훨씬 높습니다. 즉, 더 좋은 조건(더 높은 연봉)의 회사에 들어갈 확률을 높여주는 역할을 합니다.\n\n### 3. 법정 선임 자격 (가장 큰 금전적 가치)\n특정 분야(전기, 소방, 